# Data Preparation — Edible Wild Plants
> Downloads the Kaggle dataset and creates an 80/10/10 train/val/test split.
> Run this notebook **once** before training any model.

In [1]:
!pip install kagglehub -q

import kagglehub, os, gc, random, shutil
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Ready.")

Ready.


In [2]:
# Mount Google Drive so the split survives Colab restarts
from google.colab import drive
drive.mount('/content/drive')

# Drive directory for models (shared across all training notebooks)
DRIVE_MODEL_DIR = '/content/drive/MyDrive/edible_plants_models'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print("Drive model dir:", DRIVE_MODEL_DIR)

Mounted at /content/drive
Drive model dir: /content/drive/MyDrive/edible_plants_models


In [3]:
# Download dataset (~1 GB, cached on subsequent runs)
path = kagglehub.dataset_download("gverzea/edible-wild-plants")
print("Dataset path:", path)
print("Top-level contents:", os.listdir(path))

Using Colab cache for faster access to the 'edible-wild-plants' dataset.
Dataset path: /kaggle/input/edible-wild-plants
Top-level contents: ['datasets', 'final_model_weights.hdf5', 'edible wild plants metadata.xls', 'vanilla_model_weights.hdf5']


In [4]:
# Locate the directory that contains the 62 class sub-folders
def find_class_root(base):
    for root, dirs, files in os.walk(base):
        if len(dirs) > 50:
            sample = os.path.join(root, dirs[0])
            if os.path.isdir(sample):
                imgs = [f for f in os.listdir(sample)
                        if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
                if imgs:
                    return root
    return None

SOURCE_DIR = find_class_root(path)

if SOURCE_DIR is None:
    raise RuntimeError("Could not find class root inside dataset path.")

all_classes = sorted([d for d in os.listdir(SOURCE_DIR)
                       if os.path.isdir(os.path.join(SOURCE_DIR, d))])

print(f"SOURCE_DIR : {SOURCE_DIR}")
print(f"Classes    : {len(all_classes)}")
print(f"First 5    : {all_classes[:5]}")

SOURCE_DIR : /kaggle/input/edible-wild-plants/datasets/dataset
Classes    : 62
First 5    : ['Alfalfa', 'Asparagus', 'Blue Vervain', 'Broadleaf Plantain', 'Bull Thistle']


In [5]:
# Create stratified 80 / 10 / 10 train / val / test split
BASE_DIR  = '/content/plant_data'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR   = os.path.join(BASE_DIR, 'val')
TEST_DIR  = os.path.join(BASE_DIR, 'test')

# Always rebuild from scratch to avoid stale data
if os.path.exists(BASE_DIR):
    shutil.rmtree(BASE_DIR)
    print("Removed old split")

def make_split(source, base, classes, seed=42, train_r=0.80, val_r=0.10):
    random.seed(seed)
    skipped = []
    counts  = {'train': 0, 'val': 0, 'test': 0}
    for cls in classes:
        src = os.path.join(source, cls)
        if not os.path.isdir(src):
            continue
        imgs = [f for f in os.listdir(src)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
                and os.path.isfile(os.path.join(src, f))]
        if len(imgs) < 5:          # skip tiny classes
            skipped.append(cls); continue
        random.shuffle(imgs)
        n_tr  = int(len(imgs) * train_r)
        n_val = max(int(len(imgs) * val_r), 1)
        for split, flist in [('train', imgs[:n_tr]),
                               ('val',   imgs[n_tr:n_tr+n_val]),
                               ('test',  imgs[n_tr+n_val:])]:
            dest = os.path.join(base, split, cls)
            os.makedirs(dest, exist_ok=True)
            for f in flist:
                shutil.copy2(os.path.join(src, f), os.path.join(dest, f))
            counts[split] += len(flist)
    return counts, skipped

print("Creating 80/10/10 split …")
counts, skipped = make_split(SOURCE_DIR, BASE_DIR, all_classes)
if skipped:
    print(f"Skipped {len(skipped)} classes (too few images): {skipped}")
print("Split counts:", counts)

Creating 80/10/10 split …
Split counts: {'train': 5246, 'val': 655, 'test': 657}


In [6]:
# Verify the split
print("\nSplit verification:")
print(f"{'Split':<8} {'Classes':>8} {'Images':>8}")
print("-" * 28)
for split, d in [('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)]:
    cls_list = [c for c in os.listdir(d) if os.path.isdir(os.path.join(d, c))]
    n_imgs   = sum(len(os.listdir(os.path.join(d, c))) for c in cls_list)
    print(f"{split:<8} {len(cls_list):>8} {n_imgs:>8}")

NUM_CLASSES = len([c for c in os.listdir(TRAIN_DIR)
                   if os.path.isdir(os.path.join(TRAIN_DIR, c))])
print(f"\nNUM_CLASSES = {NUM_CLASSES}")
if NUM_CLASSES != 62:
    print(f"WARNING: Expected 62 classes, got {NUM_CLASSES}")
else:
    print("All 62 classes present — data preparation complete!")


Split verification:
Split     Classes   Images
----------------------------
train          62     5246
val            62      655
test           62      657

NUM_CLASSES = 62
All 62 classes present — data preparation complete!
